In [ ]:
!pip install paho-mqtt plotly pandas requests scikit-learn

In [ ]:
# Jika belum install scikit-learn, jalankan: !pip install paho-mqtt plotly pandas requests scikit-learn

import paho.mqtt.client as mqtt
import json, ssl, time, requests
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
from IPython.display import clear_output
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ========================
# KONFIGURASI EMQX CLOUD
# ========================
BROKER      = "y16d2f91.ala.asia-southeast1.emqxsl.com"
PORT        = 8883
USERNAME    = "adamnirvan"
PASSWORD    = "orichicrist"
TOPIC_TELEM = "iot/esp32-lab-01/telemetry"
TOPIC_CMD   = "iot/esp32-lab-01/cmd/led"

# ========================
# KONFIGURASI TELEGRAM & AMBANG BATAS KARHUTLA
# ========================
TELEGRAM_TOKEN   = "9189197123:AAGkfzG_AKJagm5D_M5HBTbx3Uxzwc8fKqk"
TELEGRAM_CHAT_ID = "5867719143"
SUHU_ALERT       = 32.0
AQI_ALERT        = 101       # Ambang batas AQI berbahaya (Unhealthy)
ALERT_COOLDOWN   = 60
last_alert_time  = 0
last_update_id   = 0

# ========================
# STATE & LOGGING
# ========================
data = []
MAX_POINTS = 50
CSV_FILENAME = "iot_karhutla_telemetry_log.csv"

# ========================
# FUNGSI TELEGRAM (NOTIFIKASI & KONTROL 2 ARAH)
# ========================
def send_telegram_msg(text):
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    try:
        requests.post(url, json={
            "chat_id": TELEGRAM_CHAT_ID,
            "text": text,
            "parse_mode": "HTML"
        }, timeout=4)
    except Exception as e:
        print("[TELEGRAM] Error:", e)

def check_telegram_commands(mqtt_client):
    """Membaca pesan masuk dari pengguna di Telegram untuk mengontrol Wokwi."""
    global last_update_id
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/getUpdates"
    try:
        resp = requests.get(url, params={"offset": last_update_id + 1, "timeout": 1}, timeout=3)
        if resp.status_code != 200:
            return
        updates = resp.json().get("result", [])
        for upd in updates:
            last_update_id = upd["update_id"]
            msg = upd.get("message", {})
            text = msg.get("text", "").strip().lower()
            if not text:
                continue

            if text == "/status" and len(data) > 0:
                last = data[-1]
                reply = (
                    "📊 <b>STATUS TERKINI SMART ENVIRONMENT (KARHUTLA)</b>\n\n"
                    f"🌡️ Suhu: <b>{last['temperature']} °C</b> | Kelembaban: <b>{last['humidity']} %</b>\n"
                    f"🌫️ Asap PM2.5: <b>{last['pm25']} µg/m³</b>\n"
                    f"🏭 Gas CO2: <b>{last['co2_ppm']} ppm</b>\n"
                    f"📈 AQI Gabungan: <b>{last['aqi']} ({last['air_status']})</b>\n"
                    f"🚨 Status Asap: <b>{last['smoke_status']}</b>\n"
                    f"💡 LED: <b>{last['led']}</b> | 🪟 Ventilasi: <b>{last['vent']} ({last['servo_angle']}°)</b>\n"
                    f"⚙️ Mode Sistem: <b>{last['mode']}</b>\n"
                    f"🤖 Klaster AI: <b>{last.get('cluster_label', 'Menunggu data...')}</b>"
                )
                send_telegram_msg(reply)
            elif text == "/led_on":
                mqtt_client.publish(TOPIC_CMD, "ON")
                send_telegram_msg("✅ Perintah dikirim: <b>LED ON (Mode MANUAL)</b>")
            elif text == "/led_off":
                mqtt_client.publish(TOPIC_CMD, "OFF")
                send_telegram_msg("✅ Perintah dikirim: <b>LED OFF (Mode MANUAL)</b>")
            elif text == "/vent_open":
                mqtt_client.publish(TOPIC_CMD, "VENT_OPEN")
                send_telegram_msg("🪟 Perintah dikirim: <b>Buka Penuh Ventilasi 180° (MANUAL)</b>")
            elif text == "/vent_close":
                mqtt_client.publish(TOPIC_CMD, "VENT_CLOSE")
                send_telegram_msg("🪟 Perintah dikirim: <b>Tutup Rapat Ventilasi 0° (MANUAL)</b>")
            elif text == "/auto":
                mqtt_client.publish(TOPIC_CMD, "MODE_AUTO")
                send_telegram_msg("🔄 Sistem dikembalikan ke <b>Mode AUTO</b>")
            elif text == "/help" or text == "/start":
                send_telegram_msg(
                    "🤖 <b>Daftar Perintah Bot IoT Karhutla:</b>\n"
                    "/status - Cek kondisi sensor & AQI terkini\n"
                    "/led_on - Nyalakan LED (Manual)\n"
                    "/led_off - Matikan LED (Manual)\n"
                    "/vent_open - Buka penuh ventilasi (180°)\n"
                    "/vent_close - Tutup rapat ventilasi (0°)\n"
                    "/auto - Kembalikan ke mode otomatis"
                )
    except Exception:
        pass

# ========================
# MACHINE LEARNING: K-MEANS CLUSTERING & ANOMALY
# ========================
def run_ml_analysis(df):
    """Mengelompokkan kondisi polusi udara Karhutla dengan K-Means & Z-Score Anomaly."""
    df["cluster"] = 0
    df["cluster_label"] = "Mengumpulkan Data..."
    df["anomaly"] = False

    if len(df) >= 6:
        # Fitur analisis mencakup suhu, kelembaban, pm25, co2, dan aqi
        features = df[["temperature", "humidity", "pm25", "co2_ppm", "aqi"]].values
        scaler = StandardScaler()
        scaled = scaler.fit_transform(features)

        # K-Means Clustering (3 Klaster)
        n_clusters = min(3, len(df))
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        df["cluster"] = kmeans.fit_predict(scaled)

        # Pelabelan klaster khusus mitigasi karhutla
        label_map = {}
        for c in range(n_clusters):
            sub = df[df["cluster"] == c]
            if sub["aqi"].mean() >= 101 or sub["pm25"].mean() >= 35.5:
                label_map[c] = "Bahaya / Asap Pekat"
            elif sub["aqi"].mean() >= 51 or sub["pm25"].mean() >= 12.1:
                label_map[c] = "Waspada / Asap Tipis"
            else:
                label_map[c] = "Optimal / Udara Bersih"
        df["cluster_label"] = df["cluster"].map(label_map)

        # Deteksi Anomali dengan Z-Score (> 2.3 standar deviasi)
        z_scores = np.abs((features - np.mean(features, axis=0)) / (np.std(features, axis=0) + 1e-5))
        df["anomaly"] = (z_scores > 2.3).any(axis=1)

    return df

# ========================
# CALLBACK MQTT
# ========================
def on_connect(client, userdata, flags, reason_code, properties=None):
    print("Terhubung ke broker MQTT, rc =", reason_code)
    client.subscribe(TOPIC_TELEM)
    print("Subscribe topik:", TOPIC_TELEM)

def on_message(client, userdata, msg):
    global last_alert_time
    try:
        payload = json.loads(msg.payload.decode())
        payload["timestamp"] = datetime.now()

        # Ekstraksi parameter baru dari ESP32
        temp       = payload.get("temperature", 0.0)
        hum        = payload.get("humidity", 0.0)
        light      = payload.get("light", 0)
        pm25       = payload.get("pm25", 0.0)
        co2_ppm    = payload.get("co2_ppm", 400)
        env_mult   = payload.get("env_multiplier", 1.0)
        aqi        = payload.get("aqi", 0)
        smoke_st   = payload.get("smoke_status", "AMAN")
        air_stat   = payload.get("air_status", "BAIK")
        led_stat   = payload.get("led", "OFF")
        vent_st    = payload.get("vent", "CLOSED")
        servo_ang  = payload.get("servo_angle", 0)
        mode_st    = payload.get("mode", "AUTO")

        data.append(payload)
        if len(data) > MAX_POINTS:
            data.pop(0)

        # Auto-logging ke file CSV Google Colab
        pd.DataFrame([payload]).to_csv(
            CSV_FILENAME, mode="a", header=not pd.io.common.file_exists(CSV_FILENAME), index=False
        )

        # Peringatan Otomatis ke Telegram jika AQI Buruk atau Suhu Terlalu Panas
        now_ts = time.time()
        if (aqi >= AQI_ALERT or temp > SUHU_ALERT) and (now_ts - last_alert_time >= ALERT_COOLDOWN):
            last_alert_time = now_ts
            pesan = (
                "⚠️ <b>PERINGATAN DARURAT KARHUTLA KALIMANTAN!</b> ⚠️\n\n"
                f"🌡️ <b>Suhu:</b> {temp} °C | 💧 <b>Kelembaban:</b> {hum}%\n"
                f"🌫️ <b>Asap PM2.5:</b> {pm25} µg/m³ (<b>{smoke_st}</b>)\n"
                f"🏭 <b>Gas CO2:</b> {co2_ppm} ppm\n"
                f"📈 <b>AQI Gabungan:</b> {aqi} (<b>{air_stat}</b>)\n"
                f"🪟 <b>Ventilasi:</b> {vent_st} ({servo_ang}°)\n\n"
                "<i>Ventilasi otomatis ditutup rapat untuk blokir asap. Gunakan /vent_open jika ingin override.</i>"
            )
            send_telegram_msg(pesan)

    except Exception as e:
        print("Gagal parsing payload MQTT:", e)

# ========================
# KONEKSI MQTT
# ========================
try:
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
except AttributeError:
    client = mqtt.Client()

client.username_pw_set(USERNAME, PASSWORD)
client.tls_set(cert_reqs=ssl.CERT_NONE)
client.tls_insecure_set(True)
client.on_connect = on_connect
client.on_message = on_message

client.connect(BROKER, PORT, 60)
client.loop_start()

print("Menunggu data telemetri pertama dari ESP32...")
while len(data) < 1:
    time.sleep(0.5)

# ========================
# FUNGSI RENDER DASHBOARD PLOTLY (4 SUBPLOT)
# ========================
def render_dashboard(df):
    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        subplot_titles=(
            "1. Suhu & Kelembaban Udara (DHT22)",
            "2. Partikulat Asap (PM2.5) & Gas CO2",
            "3. Indeks Kualitas Udara (AQI) & Deteksi Anomali",
            "4. Klaster AI K-Means: Mitigasi Karhutla"
        ),
        vertical_spacing=0.08,
        specs=[[{"secondary_y": True}], [{"secondary_y": True}], [{"secondary_y": False}], [{"secondary_y": False}]]
    )

    # Baris 1: Suhu & Kelembaban
    fig.add_trace(go.Scatter(x=df["timestamp"], y=df["temperature"], mode="lines+markers", name="Suhu (°C)", line=dict(color="red")), row=1, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=df["timestamp"], y=df["humidity"], mode="lines+markers", name="Kelembaban (%)", line=dict(color="blue")), row=1, col=1, secondary_y=True)

    # Baris 2: PM2.5 & CO2
    fig.add_trace(go.Scatter(x=df["timestamp"], y=df["pm25"], mode="lines+markers", name="PM2.5 (µg/m³)", line=dict(color="darkorange")), row=2, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=df["timestamp"], y=df["co2_ppm"], mode="lines+markers", name="CO2 (ppm)", line=dict(color="brown")), row=2, col=1, secondary_y=True)

    # Baris 3: AQI + Titik Anomali Z-Score
    fig.add_trace(go.Scatter(x=df["timestamp"], y=df["aqi"], mode="lines+markers", name="AQI Gabungan", line=dict(color="purple")), row=3, col=1)
    anomalies = df[df["anomaly"] == True]
    if not anomalies.empty:
        fig.add_trace(go.Scatter(
            x=anomalies["timestamp"], y=anomalies["aqi"],
            mode="markers", name="Anomali Terdeteksi",
            marker=dict(color="black", size=12, symbol="x")
        ), row=3, col=1)

    # Baris 4: Klaster K-Means AI
    fig.add_trace(go.Scatter(
        x=df["timestamp"], y=df["cluster"],
        mode="lines+markers", name="Klaster K-Means",
        text=df["cluster_label"],
        marker=dict(size=9, color=df["cluster"], colorscale="Viridis", showscale=False),
        line=dict(color="teal", dash="dot")
    ), row=4, col=1)

    # Garis Batas Ambang (Thresholds)
    fig.add_hline(y=SUHU_ALERT, line_dash="dot", line_color="red", row=1, col=1)
    fig.add_hline(y=35.5, line_dash="dot", line_color="darkorange", row=2, col=1) # Batas PM2.5 Unhealthy
    fig.add_hline(y=101, line_dash="dot", line_color="purple", row=3, col=1)   # Batas AQI Unhealthy

    # Anotasi Header Informasi Real-Time di Atas Grafik
    last = df.iloc[-1]
    air_color = "green" if last["air_status"] == "BAIK" else ("orange" if last["air_status"] == "SEDANG" else "red")
    smoke_color = "green" if last["smoke_status"] == "AMAN" else ("orange" if last["smoke_status"] == "TIPIS" else "red")

    fig.add_annotation(
        text=(
            f"Mode: <b>{last['mode']}</b> | "
            f"LED: <b>{last['led']}</b> | "
            f"Ventilasi: <b>{last['vent']} ({last['servo_angle']}°)</b> | "
            f"Asap: <b style='color:{smoke_color}'>{last['smoke_status']}</b> | "
            f"AQI: <b style='color:{air_color}'>{last['aqi']} ({last['air_status']})</b> | "
            f"AI: <b>{last['cluster_label']}</b>"
        ),
        xref="paper", yref="paper",
        x=0.5, y=1.06,
        showarrow=False,
        font=dict(size=11),
        bgcolor="white", bordercolor="gray", borderwidth=1
    )

    fig.update_layout(
        height=950,
        title="AIoT Karhutla Kalimantan Dashboard — Real-Time Telemetry, Multi-Sensor & K-Means",
        hovermode="x unified",
        legend=dict(orientation="h", y=1.10)
    )
    fig.show()

# ========================
# LOOP UTAMA (GRAFIK + POLLING BOT TELEGRAM)
# ========================
last_plotted_time = None

try:
    while True:
        # Periksa perintah masuk dari Telegram
        check_telegram_commands(client)

        current_latest_time = data[-1]["timestamp"]
        if current_latest_time != last_plotted_time:
            last_plotted_time = current_latest_time
            df = pd.DataFrame(data)
            df = run_ml_analysis(df)
            data[-1]["cluster_label"] = df.iloc[-1]["cluster_label"]

            clear_output(wait=True)
            render_dashboard(df)
            print(f"Update: {current_latest_time.strftime('%H:%M:%S')} | Log tersimpan di: {CSV_FILENAME}")
            print("Ketik /help atau /status di Bot Telegram Anda untuk memantau atau mengontrol Wokwi!")

        time.sleep(1)

except KeyboardInterrupt:
    print("\nPemantauan Google Colab dihentikan.")
    client.loop_stop()
    client.disconnect()

Update: 19:36:52 | Log tersimpan di: iot_karhutla_telemetry_log.csv
Ketik /help atau /status di Bot Telegram Anda untuk memantau atau mengontrol Wokwi!

Pemantauan Google Colab dihentikan.


In [ ]:
# Kirim perintah LED dari Colab
client.publish("iot/esp32-lab-01/cmd/led", "ON")
print("Perintah ON terkirim")